# Kaggle player-band replay harvest

This notebook downloads publicly available replays from selected leaderboard bands for the Pokémon TCG AI Battle Challenge. It compares top, middle, and lower-ranked players and creates a replay-derived decision dataset for research.

The notebook does not store credentials, alter `submission/main.py`, or claim that every recorded action is an optimal label. Replay access and leaderboard fields can vary by competition and account permissions; cells skip unavailable items and report them.

**Setup:** install `kaggle`, `pandas`, and optionally `scikit-learn`, then configure Kaggle authentication using `~/.kaggle/kaggle.json`, `KAGGLE_API_TOKEN`, or the normal Kaggle notebook identity.

In [ ]:
# If needed, run once in a notebook cell:
%pip install -q kaggle pandas scikit-learn

In [ ]:
from pathlib import Path
import json, time, hashlib
from collections import Counter

COMPETITION = 'pokemon-tcg-ai-battle'
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
OUTPUT_DIR = (REPO_ROOT / 'data' / 'player_bands').resolve()
TOP_K = 5
MIDDLE_K = 5
BOTTOM_K = 5
MAX_EPISODES_PER_PLAYER = 100
REQUEST_DELAY_SECONDS = 0.25
RETRY_COUNT = 3
DOWNLOAD_REPLAYS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)

In [ ]:
import os
from pathlib import Path

# Kaggle's newer CLI stores a token here. It is read locally and never printed or saved.
access_token = Path.home() / '.kaggle' / 'access_token'
if access_token.exists():
    os.environ.setdefault('KAGGLE_API_TOKEN', access_token.read_text().strip())

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print('Kaggle authentication succeeded')

## Resolve leaderboard bands

Leaderboard entries do not expose exactly the same attributes in every Kaggle API release. The helper below accepts common field names and falls back to matching public submissions by team name.

In [ ]:
def field(obj, *names, default=None):
    for name in names:
        value = getattr(obj, name, None)
        if value not in (None, ''):
            return value
    return default

def numeric(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

leaderboard = list(api.competition_leaderboard_view(COMPETITION))
rows = []
for i, entry in enumerate(leaderboard, start=1):
    score = numeric(field(entry, 'score', 'public_score'))
    rank = field(entry, 'rank', 'team_rank', default=i)
    rows.append({
        'rank': int(rank) if str(rank).isdigit() else i,
        'team_name': str(field(entry, 'team_name', 'teamName', 'name', default='unknown')),
        'score': score,
        'team_id': field(entry, 'team_id', 'teamId', '_team_id'),
        'submission_id': field(entry, 'submission_id', 'submissionId', 'ref', 'id'),
    })
rows = [r for r in rows if r['score'] is not None]
rows.sort(key=lambda r: (r['rank'], -r['score']))

# Leaderboard entries commonly omit submission IDs. Resolve them through each public team.
for r in rows:
    if r['submission_id'] is None and r.get('team_id') is not None:
        try:
            team_subs = list(api.competition_team_submissions(int(r['team_id'])))
            if team_subs:
                r['submission_id'] = field(team_subs[0], 'id', 'ref', 'submission_id', '_id')
        except Exception as exc:
            r['resolve_error'] = str(exc)

# Older API versions may require the team-name fallback.
if any(r['submission_id'] is None for r in rows):
    try:
        submissions = list(api.competition_submissions(COMPETITION))
        by_name = {}
        for sub in submissions:
            name = str(field(sub, 'team_name', 'teamName', 'owner_name', 'author', default=''))
            sid = field(sub, 'ref', 'id', 'submission_id')
            if name and sid is not None:
                by_name.setdefault(name, sid)
        for r in rows:
            r['submission_id'] = r['submission_id'] or by_name.get(r['team_name'])
    except Exception as exc:
        print('Submission fallback unavailable:', exc)

usable = [r for r in rows if r['submission_id'] is not None]
if not usable:
    raise RuntimeError('No leaderboard rows with usable submission IDs were returned.')

n = len(usable)
top = usable[:TOP_K]
mid_start = max(0, (n - MIDDLE_K) // 2)
middle = usable[mid_start:mid_start + MIDDLE_K]
bottom = usable[-BOTTOM_K:]
selected = []
seen = set()
for band, band_rows in [('top', top), ('middle', middle), ('bottom', bottom)]:
    for r in band_rows:
        sid = str(r['submission_id'])
        if sid not in seen:
            selected.append({**r, 'band': band})
            seen.add(sid)

print(f'Leaderboard rows: {len(rows)}; selected players: {len(selected)}')
display(selected)

In [ ]:
# Save the exact selection so later runs remain auditable.
(OUTPUT_DIR / 'selected_players.json').write_text(json.dumps(selected, indent=2, default=str))

def cached_replay_path(episode_id):
    return OUTPUT_DIR / 'replays' / f'episode-{episode_id}-replay.json'

def download_episode(episode_id):
    path = cached_replay_path(episode_id)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return path
    last_error = None
    for attempt in range(RETRY_COUNT):
        try:
            api.competition_episode_replay(int(episode_id), path=str(path.parent))
            # Kaggle normally writes the canonical filename; tolerate an API variant.
            candidates = list(path.parent.glob(f'*{episode_id}*replay.json'))
            if candidates and candidates[0] != path:
                candidates[0].replace(path)
            if path.exists():
                time.sleep(REQUEST_DELAY_SECONDS)
                return path
        except Exception as exc:
            last_error = exc
            time.sleep(1 + attempt)
    print(f'  replay {episode_id} unavailable: {last_error}')
    return None

episode_rows, failures = [], []
for player in selected:
    sid = int(player['submission_id'])
    try:
        episodes = list(api.competition_list_episodes(sid))[:MAX_EPISODES_PER_PLAYER]
    except Exception as exc:
        failures.append({'submission_id': sid, 'error': str(exc)})
        continue
    print(player['band'], player['rank'], player['team_name'], ':', len(episodes), 'episodes')
    for ep in episodes:
        eid = field(ep, 'id', 'episode_id')
        if eid is None: continue
        target_agent = next((a for a in (getattr(ep, 'agents', None) or [])
                             if str(field(a, 'submission_id')) == str(sid)), None)
        agent_index = field(target_agent, 'index', default=None)
        episode_rows.append({**player, 'episode_id': int(eid), 'agent_index': agent_index})
        if DOWNLOAD_REPLAYS: download_episode(eid)

# Deduplicate when a player appears in overlapping leaderboard bands or episode listings.
episode_rows = list({(r['submission_id'], r['episode_id']): r for r in episode_rows}.values())
(OUTPUT_DIR / 'episode_index.json').write_text(json.dumps(episode_rows, indent=2, default=str))
(OUTPUT_DIR / 'download_failures.json').write_text(json.dumps(failures, indent=2))
print('Indexed episodes:', len(episode_rows), '| failures:', len(failures))

## Convert replays to decision records

A replay step contains the observation before a decision, while the following step contains the resulting action. The converter therefore pairs `steps[t][player].observation` with `steps[t+1][player].action`. This avoids the common one-step label shift. Deck-selection actions are excluded from the behavior records.

In [ ]:
def decks_from_replay(replay):
    decks = {}
    for step in replay.get('steps', []):
        for i, agent in enumerate(step):
            action = agent.get('action')
            if isinstance(action, list) and len(action) == 60 and i not in decks:
                decks[i] = action
        if len(decks) >= 2: break
    return decks

def team_indices(replay, team_name):
    info = replay.get('info', {})
    names = info.get('TeamNames') or [a.get('Name') for a in info.get('Agents', [])]
    return [i for i, name in enumerate(names or []) if name == team_name]

records, stats = [], Counter()
for item in episode_rows:
    path = cached_replay_path(item['episode_id'])
    if not path.exists(): stats['missing_replay'] += 1; continue
    try: replay = json.loads(path.read_text())
    except Exception: stats['bad_json'] += 1; continue
    rewards = replay.get('rewards') or []
    steps = replay.get('steps') or []
    # Prefer the recorded player index from the episode listing; infer by name if absent.
    target_index = item.get('agent_index')
    if target_index is None:
        idxs = team_indices(replay, item['team_name'])
        target_index = idxs[0] if idxs else None
    if target_index is None: stats['player_not_found'] += 1; continue
    target_index = int(target_index)
    outcome = rewards[target_index] if target_index < len(rewards) else None
    if outcome not in (-1, 1): stats['draw_or_unknown'] += 1; continue
    decks = decks_from_replay(replay)
    for t, step in enumerate(steps[:-1]):
        if target_index >= len(step) or target_index >= len(steps[t + 1]): continue
        agent = step[target_index]
        obs, next_agent = agent.get('observation'), steps[t + 1][target_index]
        action = next_agent.get('action')
        if agent.get('status') != 'ACTIVE' or not isinstance(obs, dict) or obs.get('select') is None: continue
        if not isinstance(action, list) or len(action) == 60: continue
        select = obs['select']
        records.append({
            'episode_id': item['episode_id'], 'submission_id': item['submission_id'],
            'band': item['band'], 'rank': item['rank'], 'team_name': item['team_name'],
            'step': t, 'outcome': int(outcome),
            'context': select.get('context'), 'min_count': select.get('minCount'),
            'max_count': select.get('maxCount'), 'option_count': len(select.get('option') or []),
            'action': action, 'observation': obs,
            'own_deck': decks.get(target_index), 'opponent_deck': decks.get(1 - target_index),
        })
        stats['records'] += 1

with (OUTPUT_DIR / 'decision_records.jsonl').open('w') as f:
    for record in records: f.write(json.dumps(record, separators=(',', ':')) + '\n')
print('Records:', len(records), dict(stats))

In [ ]:
import pandas as pd

df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('observation', 'action', 'own_deck', 'opponent_deck')} for r in records])
if len(df):
    print('Records by band')
    display(df.groupby('band').agg(records=('episode_id','size'), games=('episode_id','nunique'), win_rate=('outcome', lambda x: (x == 1).mean())))
    print('Most common decision contexts')
    display(df.groupby(['band', 'context']).size().reset_index(name='records').sort_values('records', ascending=False).head(30))
    print('Rank-band coverage')
    display(df.groupby(['band', 'rank']).size().reset_index(name='records'))
else:
    print('No records were created; inspect download_failures.json and API access.')

## Optional lightweight baseline

This is only a diagnostic: it predicts the final outcome from coarse replay metadata and reports held-out accuracy. It is not a replacement for the game agent and does not write model code into the submission.

In [ ]:
try:
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import accuracy_score, roc_auc_score
    if len(df) >= 20 and df['outcome'].nunique() == 2:
        features = ['band', 'context', 'option_count', 'step']
        X, y = df[features], (df['outcome'] == 1).astype(int)
        groups = df['episode_id']
        train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42).split(X, y, groups))
        prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['band', 'context']), ('num', 'passthrough', ['option_count', 'step'])])
        model = make_pipeline(prep, LogisticRegression(max_iter=500))
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx]); prob = model.predict_proba(X.iloc[test_idx])[:, 1]
        print('Episode-held-out accuracy:', round(accuracy_score(y.iloc[test_idx], pred), 4))
        print('Episode-held-out ROC AUC:', round(roc_auc_score(y.iloc[test_idx], prob), 4))
    else:
        print('Need at least 20 records and both outcomes for the optional baseline.')
except ImportError:
    print('scikit-learn is not installed; skipping optional baseline.')

## Outputs

- `selected_players.json`: the leaderboard rows and selected bands.
- `episode_index.json`: episode IDs associated with each selected player.
- `replays/`: cached replay JSON files.
- `decision_records.jsonl`: observations, aligned actions, outcomes, bands, and deck information.
- `download_failures.json`: items unavailable to the authenticated account.

For model training, split by episode rather than individual decisions to prevent one game from appearing in both training and validation. Treat top-player actions as behavior examples, not automatically correct actions, and compare any learned policy against the current agent in local evaluation before adopting it.